# 1. 데이터 분석

`4G100 CUP (CV5)` 의 주물사 시험값과 불량 집계를 살펴본다.
전처리·학습은 다음 노트북에서 다룬다.

In [ ]:
import sys, os, warnings
sys.path.insert(0, '/data/Project/boyoung/pipeline')
for v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(v, '1')
warnings.filterwarnings('ignore')

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
plt.rc('font', family='NanumGothic'); plt.rc('axes', unicode_minus=False)
sns.set_theme(style='whitegrid', font='NanumGothic', rc={'axes.unicode_minus': False})
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 40)

import db, preprocess as P, classify as C

print('DB 동기화 상태'); db.freshness()

## 1.1 원본 구성

In [ ]:
f = db.table('foundry_test_result')
g = f[f['product_name'] == P.PRODUCT].copy()
b = db.table('bad_statistics')

print(f'주물사 시험  전체 {len(f):,}행 · 제품 {f["product_name"].nunique()}종')
print(f'             해당 제품 {len(g)}행')
print(f'불량 집계    {len(b)}행 · 제품 {b["product_name"].nunique()}종  ← 이 제품만 있다')
print()
print('로트당 시험 횟수')
print(g.assign(k=P.lot_key(g['lot_no'])).groupby('k').size().value_counts().sort_index().to_string())

## 1.2 측정 항목 분포

In [ ]:
pt = g['bent_val'].astype(str).str.extract(r'^\s*(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)\s*$')
g['bentonite'] = pd.to_numeric(pt[0], errors='coerce')

COLS = [('bentonite','벤토나이트','—'), ('sand_temp','모래온도','℃'), ('moist_val','수분','%'),
        ('fill_val','충진율','%'), ('cstr_val','압축강도','kPa'), ('sstr_val','전단강도','kPa'),
        ('perm_val','통기도','—'), ('fswt_val','충진시편','g'), ('sswt_val','표준시편','g')]
rows = []
for c, n, u in COLS:
    s = pd.to_numeric(g[c], errors='coerce').dropna()
    rows.append({'항목': n, '단위': u, '개수': len(s), '평균': s.mean(), '표준편차': s.std(),
                 '최소': s.min(), '25%': s.quantile(.25), '중앙값': s.median(),
                 '75%': s.quantile(.75), '최대': s.max()})
pd.DataFrame(rows).round(2)

**변동 폭이 항목마다 크게 다르다.** 수분은 표준편차 0.25 로 거의 고정에 가깝고, 통기도는 29.27 로 크게 흔들린다.

## 1.3 타겟 — 불량률

In [ ]:
data, report = P.build()
data = data.sort_values('target_date').reset_index(drop=True)
th = C.threshold(data['bad_rate'])
y = C.label(data['bad_rate'], th)
print(report)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
sns.histplot(data['bad_rate'], bins=22, kde=True, ax=ax[0], color='#b8531c')
ax[0].axvline(th, color='#0b6b6b', lw=2, label=f'평균 {th:.2f}%')
ax[0].set_xlabel('불량률 (%)'); ax[0].legend(fontsize=9); ax[0].set_title('불량률 분포')
sns.boxplot(y=data['bad_rate'], ax=ax[1], color='#7aa6c2'); ax[1].set_title('상자그림')
sns.scatterplot(data=data, x='production_qty', y='bad_rate', ax=ax[2], color='#b8531c', s=28)
ax[2].set_title('생산수량 ↔ 불량률'); ax[2].set_xlabel('생산수량')
plt.tight_layout(); plt.show()

print(data[['bad_rate','production_qty','bad_qty','n_tests','night_ratio']]
      .describe().loc[['mean','std','min','50%','max']].round(2).to_string())

생산수량이 1,900~4,700 으로 2.5배 차이난다. **불량 개수를 그대로 쓰면 생산량을 학습**하므로 비율로 바꾼다.

## 1.4 결측치

In [ ]:
print('원본 (해당 제품)')
na = g[['lot_no','real_date','bent_val','sand_temp'] + [f'{p}_val' for p in P.SPEC]].isna().sum()
print(na[na > 0].to_string() if na.sum() else '  없음')
print(f'\n로트 집계 후 {len(data)}행 × 변수 {len(P.feature_columns(data))}개')
na2 = data[P.feature_columns(data)].isna().sum()
print(na2[na2 > 0].to_string() if na2.sum() else '  결측 없음')
print(f'\n시험 로트 {report["시험 로트"]} · 불량 로트 {report["불량 로트"]} → 매칭 {report["매칭"]} ({report["매칭률"]})')

## 1.5 이상치와 규격

In [ ]:
MM = {'벤토나이트':'bentonite', '모래온도':'sand_temp_mean'}
MM |= {n: f'{p}_val_mean' for p, n in P.MEASURES if p != 'sand_temp'}

rows = []
for n, c in MM.items():
    s = data[c].dropna(); q1, q3 = s.quantile([.25,.75]); i = q3 - q1
    out = int(((s < q1-1.5*i) | (s > q3+1.5*i)).sum())
    pre = c.replace('_val_mean','').replace('_mean','')
    oc = f'{pre}_out_sum'
    ex = int((data[oc] > 0).sum()) if oc in data else None
    rows.append({'항목': n, 'IQR 밖': out, 'IQR 비율': f'{100*out/len(s):.1f}%',
                 '규격 이탈 로트': ex if ex is not None else '규격 없음',
                 '이탈 비율': f'{100*ex/len(data):.0f}%' if ex is not None else ''})
print(pd.DataFrame(rows).to_string(index=False))

fig, axes = plt.subplots(2, 5, figsize=(15.5, 6))
for ax, (n, c) in zip(axes.ravel(), MM.items()):
    sns.boxplot(y=data[c], ax=ax, color='#7aa6c2', width=.5, fliersize=3.5)
    ax.set_title(n, fontsize=10); ax.set_ylabel('')
for ax in axes.ravel()[len(MM):]: ax.axis('off')
fig.suptitle('변수별 분포와 이상치', fontsize=13); fig.tight_layout(); plt.show()

**IQR 밖은 제거하지 않는다.** 불량률 상위 구간에 실제 정보가 있고, 상위 절단을 시도했으나 어느 타겟에서도 나아지지 않았다.

**규격 이탈은 변수로 만든다.** 다만 압축강도·전단강도·통기도는 규격에 **상한이 없어**(`999`) 이탈이 0건이다. 관리가 잘 돼서가 아니다.

In [ ]:
for p in P.SPEC:
    lo = g[f'{p}_min'].astype(float).mode()[0]; hi = g[f'{p}_max'].astype(float).mode()[0]
    v = pd.to_numeric(g[f'{p}_val'], errors='coerce')
    print(f'  {P.ko(p+"_val_mean"):10} 규격 {lo:>6.1f} ~ {hi:>6.1f}   실제 {v.min():.1f} ~ {v.max():.1f}')

## 1.6 상관관계

In [ ]:
cols = list(MM.values())
cm = data[cols + ['bad_rate']].rank().corr()
cm.index = cm.columns = list(MM) + ['불량률']

fig, ax = plt.subplots(figsize=(8.6, 6.8))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True,
            linewidths=.5, cbar_kws={'label':'순위 상관'}, ax=ax, annot_kws={'size':8.5})
ax.set_title('변수 간 · 변수와 불량률의 순위 상관')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout(); plt.show()

**충진율과 충진시편무게가 -0.98** 로 사실상 같은 값이다. 변수 간 상관이 높은 것은 **버릴 근거**이지 쓸 근거가 아니다.

타겟과의 상관은 **벤토나이트 0.392 · 모래온도 0.299** 둘만 0.3 을 넘고 나머지 일곱은 0.07 미만이다.

## 1.7 불량 유형

In [ ]:
rows = []
for t, n in P.DEFECTS.items():
    rows.append({'유형': n, '발생 로트': int((data[t] > 0).sum()),
                 '총 불량': int(data[t].sum()),
                 '평균 비율(%)': round(data[f'{t}_rate'].mean(), 3)})
s = pd.DataFrame(rows).sort_values('총 불량', ascending=False)
print(s.to_string(index=False))

tot = data['bad_qty'].sum(); typed = data[list(P.DEFECTS)].sum().sum()
print(f'\n총 불량 {tot:,}  유형 합계 {typed:,.0f}  미분류 {100*(tot-typed)/tot:.1f}%')

발생 로트가 15개 이하인 유형은 **0 을 답하는 모델이 최적**이 되어 배울 것이 없다.

미분류 27% 는 불량일보가 수기 작성이라 유형 칸을 비웠을 가능성이 있다.

## 1.8 계절

In [ ]:
by = data.groupby('month').agg(불량률=('bad_rate','mean'), 모래온도=('sand_temp_mean','mean'),
                               벤토나이트=('bentonite','mean'), 로트=('bad_rate','size'))
fig, ax = plt.subplots(figsize=(9, 3.6))
a2 = ax.twinx()
ax.plot(by.index, by['불량률'], marker='o', color='#b8531c', label='불량률(%)')
ax.plot(by.index, by['벤토나이트'], marker='^', color='#0b6b6b', label='벤토나이트')
a2.plot(by.index, by['모래온도'], marker='s', ls=':', color='#999')
a2.set_ylabel('모래온도 (℃)', color='#999'); a2.grid(False)
ax.set_xlabel('월'); ax.legend(fontsize=9, loc='lower left'); ax.set_title('월별 추이')
plt.tight_layout(); plt.show()
print(by.round(2).to_string())

**모래온도는 계절을 그대로 반영한다.** 1월 14.5℃ 에서 8월 37.7℃ 까지 오르고, 로트 순서와의 상관이 `+0.867` 이다.

이 때문에 무작위로 나눠 검증하면 **계절을 외운 것이 실력처럼 보인다.** 자세한 내용은 `3-모델학습` 참조.